# Recalibrate a trained supernet

Runs nothing but BN post-statistics on checkpoints that already exist.
About a minute a branch instead of three hours, because the weights are
not the thing being repaired.

Two uses:

**Repairing a run made before the calibration fix.** `bn_calibration_init`
reset the statistics and then left `training` False on the outer module,
so `batch_norm` was told not to refill them and every width normalized by
zero mean and unit variance. A run that reached 75.4% at width 1.0 during
training reported 99.0% error at all sixteen widths afterwards. The
trained weights were never affected.

**Reading an existing supernet at widths it was never evaluated at.** Put
them in `width_mult_list_test` in `apps/recal_*.yml` and run this again.

Attach every session whose output holds the checkpoints first. Branches
are located one by one, so A and B saved by one session and C and D by
another can be recalibrated together into a single table.

In [ ]:
# Every branch to recalibrate. They are located independently, so listing
# all four works as long as both sessions are attached as inputs.
BRANCHES = ['a_kl', 'b_alpha', 'c_wasserstein', 'd_wasserstein_pair']

# The directory holding the run being recalibrated. Leave it empty and the
# cell below finds it: Kaggle mounts an attached output under a slug of
# its own, which is rarely the title shown in the editor. Set it only to
# point at one particular run among several attached.
LOGS_FROM = ''

REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'

CIFAR_DIR = ('/kaggle/input/datasets/nlnk1607/cifar100/cifar-100-python')

In [ ]:
import os
import queue
import re
import shutil
import subprocess
import sys
import threading

import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)

WORK = '/kaggle/working'
CODE = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(CODE):
    subprocess.run(
        ['git', 'clone', '-b', REPO_BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)
print('cwd', os.getcwd())

In [ ]:
TARGET = 'data/cifar-100-python'
if not os.path.isdir(TARGET):
    source = CIFAR_DIR if os.path.isdir(CIFAR_DIR) else None
    if source is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'cifar-100-python' in dirs:
                source = os.path.join(root, 'cifar-100-python')
                break
    os.makedirs('data', exist_ok=True)
    if source:
        os.symlink(source, TARGET)
        print('linked', source)
    else:
        from torchvision import datasets
        datasets.CIFAR100(root='data', train=True, download=True)
        datasets.CIFAR100(root='data', train=False, download=True)
print(sorted(os.listdir(TARGET)))

In [ ]:
# The fix this repairs, checked in seconds before a card is touched. If
# this fails there is no point recalibrating.
!python tests/test_bn_calibration.py

In [ ]:
def find_checkpoints(explicit):
    """locate cifar100_<branch>/best_model.pt for each branch

    Searched by shape rather than by name. An attached run sits under a
    slug Kaggle chooses, at a depth that depends on how it was saved, and
    two runs attached at once live under two different slugs. What is
    stable is a directory named after the branch with a checkpoint in it,
    so each branch is looked up on its own rather than all of them being
    assumed to share one parent.
    """
    roots = [explicit] if explicit else []
    roots.append('/kaggle/input')
    found = {}
    for root in roots:
        if not os.path.isdir(root):
            continue
        for walked, dirs, _ in os.walk(root):
            for branch in BRANCHES:
                name = 'cifar100_{}'.format(branch)
                if branch in found or name not in dirs:
                    continue
                candidate = os.path.join(walked, name)
                if os.path.exists(
                        os.path.join(candidate, 'best_model.pt')):
                    found[branch] = candidate
    return found


found = find_checkpoints(LOGS_FROM)
missing = [branch for branch in BRANCHES if branch not in found]
if missing:
    print('no checkpoint found for', missing)
    print('what is attached:')
    for walked, dirs, _ in os.walk('/kaggle/input'):
        if walked.count('/') > 4:
            dirs[:] = []
            continue
        print('   ', walked)
    raise SystemExit(
        'attach the session whose output holds '
        'logs/cifar100_<branch>/best_model.pt, or drop that branch from '
        'BRANCHES, then run this cell again')

# Only best_model.pt is read, by pretrained in apps/recal_*.yml. The
# calibrated file beside it is the broken output being replaced and the
# optimizer state is 90 MB of no use here, so neither is copied.
for branch, src in sorted(found.items()):
    dst = 'logs/cifar100_{}'.format(branch)
    os.makedirs(dst, exist_ok=True)
    shutil.copy2(os.path.join(src, 'best_model.pt'),
                 os.path.join(dst, 'best_model.pt'))
    print('{:32} <- {}'.format(dst, src))

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+),\s+top5_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs):
    lines = queue.Queue()
    procs = {}

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for index, (label, config) in enumerate(jobs):
        env = dict(os.environ)
        env['CUDA_VISIBLE_DEVICES'] = str(index % max(n_gpu, 1))
        proc = subprocess.Popen(
            [sys.executable, '-u', 'train.py', 'app:' + config],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=env)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] started on gpu {}'.format(
            label, env['CUDA_VISIBLE_DEVICES']), flush=True)

    remaining = len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {}'.format(
                label, procs[label].returncode), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1, top5 = found.groups()
            results.setdefault(label, {})[float(width)] = (
                float(loss), float(top1), float(top5))
        if ('val' in line or 'cal' in line.split(':')[0]
                or 'Error' in line or 'Traceback' in line
                or 'Loaded' in line or 'Skip' in line):
            print('[{}] {}'.format(label, line), flush=True)

    return {label: proc.returncode for label, proc in procs.items()}



# Four jobs on two cards run two to a card. Calibration is twenty
# batches and a validation pass, so the memory two of them need together
# is nothing like a training run's.
codes = run_pinned([(b, 'apps/recal_{}.yml'.format(b)) for b in BRANCHES])
print(codes)

## The table

These are the numbers a paper would quote: accuracy at every width, with
the post-statistics recomputed for each one.

One seed. The project's own specialization gap moved by 0.73 / 2.71 / 3.15
across three seeds at a single width, so a few tenths of a point between
branches is not a difference yet.

In [ ]:
widths = sorted({w for table in results.values() for w in table})
# wide enough for the longest branch name, so a four-branch table lines
# up with its own header
column = max(14, max(len(b) for b in BRANCHES) + 2)
print('{:>7}'.format('width') + ''.join(
    '{:>{}}'.format(b, column) for b in BRANCHES))
for width in widths:
    row = '{:>7.2f}'.format(width)
    for branch in BRANCHES:
        entry = results.get(branch, {}).get(width)
        row += '{:>{}.2f}%'.format(
            100.0 * (1.0 - entry[1]), column - 1) if entry else (
                '{:>{}}'.format('-', column))
    print(row)

print()
label = max(len(b) for b in BRANCHES) + 2
for branch in BRANCHES:
    table = results.get(branch, {})
    if not table:
        continue
    accuracies = [100.0 * (1.0 - v[1]) for v in table.values()]
    print('{:{}} mean over widths {:.2f}%   worst {:.2f}%'.format(
        branch, label, sum(accuracies) / len(accuracies), min(accuracies)))

out = os.path.join(WORK, 'logs')
for branch in BRANCHES:
    log_dir = 'logs/recal_{}'.format(branch)
    if os.path.isdir(log_dir):
        shutil.copytree(log_dir, os.path.join(out, 'recal_' + branch),
                        dirs_exist_ok=True)
print('\ncalibrated checkpoints copied to', out)